In [1]:
from langchain_community.document_loaders import PyPDFLoader 
loader= PyPDFLoader('ML_ques_ans.pdf')
docs= loader.load()
docs

[Document(metadata={'producer': 'Skia/PDF m96 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': '100 Machine Learning Interview Questions and Answers', 'source': 'ML_ques_ans.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='100 Machine Learning Interview Questions andAnswers\n1. Please Explain Machine Learning, Artificial Intelligence, And Deep Learning?\nMachine learning is defined as a subset of Artificial Intelligence, and it contains the techniqueswhich enable computers to sort things out from the data and deliver Artificial Intelligenceapplications. Artificial Intelligence (AI) is a branch of computer science that is mainly focused onbuilding smart machines that can perform certain tasks that mainly require human intelligence. Itis the venture to replicate or simulate human intelligence in machines.Deep learning can be defined as a class of machine learning algorithms in Artificial Intelligencethat mainly uses multiple layers to cumulativ

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#pip install langchain-text-splitters
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
#first chunk - 0 to 1000 words 
# second chunk - 800 to 1800 words 
# third chunk - 1600 to 2600 words .... upto 

In [3]:
document= text_splitter.split_documents(docs)

In [4]:
#document

In [5]:
def clean_text(text):
    return text.encode("utf-8","ignore").decode("utf-8","ignore")

# convert text into utf-8 bytes 
# invalid characters are ignored
# .decode () convert bytes back to normal string

In [6]:
cleaned_docs=[]
for d in document:
    d.page_content= clean_text(d.page_content)
    cleaned_docs.append(d)

In [7]:
# vector DB - vector embeddings 
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS 

In [8]:
emb= OllamaEmbeddings(model='nomic-embed-text')

In [9]:
vectorstore= FAISS.from_documents(
    documents= cleaned_docs,
    embedding= emb
)

In [10]:
ret= vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={"k":5}
)

In [11]:
from langchain_core.prompts import ChatPromptTemplate 
prompt= ChatPromptTemplate.from_template(
    """
    You are an expert research assistant.
    Answer the question using only the provided context
    If the answer is not in the content, say "Not found in the context"
    context:{context}
    Question :{question}
    
    Answer: 
    """
)

In [12]:
from langchain_ollama import ChatOllama

llm= ChatOllama(
    model='gemma3:1B',
    temperature =0.2  # o to 1
)

In [14]:
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain= (
    {
        "context": ret | format_docs,
        "question":RunnablePassthrough()
    }
    |prompt
    |llm
    |StrOutputParser()
)

# Retriever ->Relevant chunks -> format chunks -> single context text
# runnablepassthrough - passes original query unchanged 
# prompt context ... question
# llm 
# stroutputparser - output in string format 

In [17]:
query = input("Enter your question")
response = rag_chain.invoke(query)

In [18]:
print(response)

Okay, here's the answer to your questions, based solely on the provided context:

**1. Explain Multilayer Perceptron And Boltzmann Machine?**

A Multilayer Perceptron (MLP) is a class of artificial neural networks that generate a set of outputs from the set of given inputs. It consists of several layers of inputnodes that are connected as a directed graph between input and output layers. The Boltzmann Machine is a type of neural network that uses a Boltzmann distribution to represent the probability of activation of each neuron.

**2. Explain The Term Bias?**

Bias refers to an error in a model that causes it to consistently under- or over-predict a value. It’s a systematic error that can distort the model's output.

**3. Explain The Term Perceptron In Machine Learning?**

A Perceptron is an algorithm for supervised learning of binary classifiers. It enables the neurons to learn and processes the elements in the given training set oneat a time. There are two types of Perceptrons, namel

In [ ]:
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_ollama import ChatOllama
# from langchain_core.output_parsers import StrOutputParser


# # Model
# llm = ChatOllama(
#     model="gemma3:1B",
#     temperature=0.7
# )


# # Prompt
# prompt = ChatPromptTemplate.from_template(
#     """
# You are a friendly interviewer.

# Ask the user one interesting question.

# After the user answers, ask another related question.

# User Answer:
# {answer}

# Next Question:
# """
# )


# # Chain
# chain = prompt | llm | StrOutputParser()


# # First question from model
# response = llm.invoke("Ask me a question.")
# print("\nModel:", response.content)


# # Chat loop
# while True:

#     user_answer = input("\nYour Answer: ")

#     if user_answer.lower() == "exit":
#         print("Chat ended.")
#         break

#     next_question = chain.invoke({

#         "answer": user_answer
#     })

#     print("\nModel:", next_question)